In [ ]:
import pandas as pd
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [ ]:
# Membuat kumpulan kalimat (korpus) yang berkaitan dengan tema Pariwisata Indonesia
daftar_stopword = StopWordRemoverFactory().get_stop_words()
alat_stemming = StemmerFactory().create_stemmer()

teks_pariwisata = [
    "Pesona matahari terbenam di Pantai Kuta sangat menakjubkan bagi para turis mancanegara.",
    "Candi Borobudur adalah peninggalan sejarah yang megah dan terletak di wilayah Jawa Tengah.",
    "Persiapan fisik sangat dibutuhkan sebelum memulai pendakian ke puncak Gunung Rinjani.",
    "Harga makanan di kawasan wisata Yogyakarta cukup terjangkau untuk kantong para pelajar.",
    "Keindahan terumbu karang di Kepulauan Raja Ampat sungguh tiada duanya di dunia laut."
]

In [ ]:

# Proses mencakup: mengubah huruf menjadi kecil, hanya menyisakan alfabet (regex), dan menghapus stopword
def praproses_teks(kalimat):
    kalimat_bersih = re.sub(r'[^a-z\s]', ' ', kalimat.lower())
    kata_mentah = kalimat_bersih.split()
    kata_tanpa_stopword = [k for k in kata_mentah if k not in daftar_stopword]
    hasil_stemming = alat_stemming.stem(' '.join(kata_tanpa_stopword))
    kata_final = hasil_stemming.split()
    return kata_mentah, kata_final

In [ ]:
# Menghitung berapa banyak penyusutan atau persentase reduksi kata antara sebelum dan sesudah diproses
laporan_reduksi = []
for i, teks in enumerate(teks_pariwisata):
    token_awal, token_akhir = praproses_teks(teks)
    jml_awal = len(token_awal)
    jml_akhir = len(token_akhir)
    persen_turun = ((jml_awal - jml_akhir) / jml_awal) * 100 if jml_awal > 0 else 0
    
    laporan_reduksi.append({
        "ID Dokumen": f"DOK-{i+1}",
        "Teks Asli": teks,
        "Token Awal": token_awal,
        "Token Proses": token_akhir,
        "Jumlah Awal": jml_awal,
        "Jumlah Akhir": jml_akhir,
        "Reduksi Dimensi (%)": round(persen_turun, 2)
    })

In [ ]:
# Loop hanya dibatasi untuk menampilkan 2 dokumen pertama sebagai sampel visual
print("=== PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING ===\n")
for item in laporan_reduksi[:2]:
    print(f">> {item['ID Dokumen']}")
    print(f"Teks Asli : {item['Teks Asli']}")
    print(f"Sebelum   : {item['Token Awal']}")
    print(f"Sesudah   : {item['Token Proses']}\n")

In [ ]:
# Menampilkan tabel statistik penyusutan (jumlah kata awal vs kata akhir) secara lebih rapi
df_laporan = pd.DataFrame(laporan_reduksi)
print("=== TABEL STATISTIK REDUKSI TOKEN ===")
display(df_laporan[['ID Dokumen', 'Jumlah Awal', 'Jumlah Akhir', 'Reduksi Dimensi (%)']])

**Evaluasi Preprocessing:**
Proses stopword removal dan stemming terbukti secara signifikan memangkas dimensi dari teks mentah. Kata-kata tidak penting dihilangkan, sedangkan kata-kata dengan imbuhan dikembalikan ke bentuk dasarnya (misalnya 'menakjubkan' menjadi 'takjub'). Pada sistem temu kembali informasi (IR), penyusutan (reduksi) token ini membuat ukuran indeks data menjadi jauh lebih kecil. Hasilnya komputasi pencarian menjadi lebih efisien dan cepat tanpa mengorbankan relevansi dokumen.